In [1]:
!pip install psycopg2

# Standard Imports

In [2]:
import os,sys
import json
from datetime import datetime
import requests


# Reading DB credential from a JSON file

A standard practice, where the DB informations are not hard coded in the script. Instead placed in file locally to access it during DB object Instantiation.

In [3]:
db_cred_dict = {
                    "host":  "ep-noisy-lake-a8k78ama-pooler.eastus2.azure.neon.tech",
                    "port" : "5432",
                    "database" : "playground",
                    "user": 'suriya',
                    "password" : 'SIt@2-3455-5',
                    "sslmode" : "require"
                }

In [4]:
with open("DB_cred.json","w") as jf:
  json.dump(db_cred_dict,jf,indent=4)

# Get weather by city

In [5]:
def get_weather_data(city):
  API_KEY = '8817d3ca9a0f4babb00182028251102'
  CITY = city
  API_URL = f'http://api.weatherapi.com/v1/current.json?key={API_KEY}&q={CITY}'
  data = requests.get(API_URL).json()
  city_name = data['location']['name']
  temperature = data['current']['temp_c']
  humidity = data['current']['humidity']
  pressure = data['current']['pressure_mb']
  weather_description = data['current']['condition']['text']
  timestamp = datetime.now()
  return {"city_name":city_name,"temperature":temperature,"humidity":humidity,"pressure":pressure,"weather_description":weather_description,"timestamp":timestamp}

# Generic DB class

This generic DB class has the following in-build methods to perform Database manipulation activities, effortlessly.

*   set_connection_details
*   get_connection_details
*get_db_version
*create_table
*insert_entry
*delete_entry
*execute_query (i.e for custom query executions)
*fetch_data (i.e for custom querying with custome conditions)



In [6]:
class db():
  def __init__(self):
    self.database = ""
    self.host = ""
    self.user = ""
    self.password = ""
    self.port = 0
    self.sslmode = ""

  def set_connection_details(self,database,host,user,password,port,sslmode):
    if self.database == "" and self.host == "" and self.user == "" and self.password == "" and self.port == 0 and self.sslmode == "":
      self.database = database
      self.host = host
      self.user = user
      self.password = password
      self.port = port
      self.sslmode = sslmode
      print("Connection set")
    else:
      print("Connection already set")

  def get_connection_details(self):
    return self.database,self.host,self.user,self.port,self.sslmode

  def get_DB_connection(self):
    database,host,user,port,sslmode = self.get_connection_details()
    try:
      import psycopg2
    except Exception as e:
      print("psycopg2 not installed, try pip install psycopg2 and try again.")

    try:
      conn = psycopg2.connect(
          host=host,
          port=port,
          database=database,
          user=user,
          password=self.password,
          sslmode=sslmode
      )
      print("Database connection established successfully.")

      cursor = conn.cursor()
      return conn,cursor

    except Exception as e:
      print(f"Error connecting to the database: {e}")
      return None,None

  def get_db_version(self):
    conn,cursor = self.get_DB_connection()
    try:
      cursor.execute("SELECT version();")
      db_version = cursor.fetchone()
      print(f"Database Version: {db_version[0]}")
    except Exception as e:
        print(f"Error connecting to the database: {e}")
    finally:
      del cursor
      conn.close()

  def create_table(self,schema_name = "",table_name = "", primary_key_name = "",columns_datatype_dict = {}):
    conn,cursor = self.get_DB_connection()
    builded_query = f"CREATE TABLE IF NOT EXISTS {schema_name}.{table_name} ({primary_key_name} SERIAL PRIMARY KEY,{', '.join([f'{key} {value}' for key, value in columns_datatype_dict.items()])});"
    try:
      cursor.execute(builded_query)
      conn.commit()
      print("Table created successfully.")
    except Exception as e:
        print(f"Error Creating table in the database: {e}")
    finally:
      del cursor
      conn.close()

  def insert_entry(self,schema_name = "",table_name = "", columns_value_dict = {}):
    conn,cursor = self.get_DB_connection()
    builded_query = f"""INSERT INTO {schema_name}.{table_name} ({', '.join(columns_value_dict.keys())}) VALUES ({', '.join([f"'{value}'" for value in columns_value_dict.values()])});"""
    try:
      cursor.execute(builded_query)
      conn.commit()
      print("Entry inserted successfully.")
    except Exception as e:
        print(f"Error inserting entry in the database: {e}")
    finally:
      del cursor
      conn.close()

  def delete_entry(self, schema_name = "", table_name = "", primary_key_name = "", primary_key_value = ""):
    conn,cursor = self.get_DB_connection()
    builded_query = f"""DELETE FROM {schema_name}.{table_name} WHERE {primary_key_name} = {primary_key_value};"""
    try:
      cursor.execute(builded_query)
      conn.commit()
      print("Entry deleted successfully.")
    except Exception as e:
        print(f"Error deleting entry in the database: {e}")
    finally:
      del cursor
      conn.close()

  def execute_query(self,query):
    conn,cursor = self.get_DB_connection()
    try:
      cursor.execute(query)
      conn.commit()
      print("Query executed successfully.")
    except Exception as e:
        print(f"Error executing query in the database: {e}")
    finally:
      del cursor
      conn.close()

  def fetch_data(self,query):
    conn,cursor = self.get_DB_connection()
    try:
      cursor.execute(query)
      data = cursor.fetchall()
      return data
    except Exception as e:
        print(f"Error executing query in the database: {e}")
    finally:
      del cursor
      conn.close()




# set_connection_details

In [7]:
new_db_obj = db()

In [8]:
new_db_cred = json.load(open("DB_cred.json","r"))

In [9]:
new_db_obj.set_connection_details(new_db_cred["database"],new_db_cred["host"],new_db_cred["user"],new_db_cred["password"],new_db_cred["port"],new_db_cred["sslmode"])

Connection set


Created a Singleton Class, the credentials can only be set once. If different connection needed to be established. Create a new object.

In [10]:
new_db_obj.set_connection_details(new_db_cred["database"],new_db_cred["host"],new_db_cred["user"],new_db_cred["password"],new_db_cred["port"],new_db_cred["sslmode"])

Connection already set


# get_connection_details

You can use this method anywhere in the program to get the DB details except the password.

In [11]:
new_db_obj.get_connection_details()

('playground',
 'ep-noisy-lake-a8k78ama-pooler.eastus2.azure.neon.tech',
 'suriya',
 '5432',
 'require')

# get_db_version

In [12]:
new_db_obj.get_db_version()

Error connecting to the database: ERROR:  Endpoint ID is not specified. Either please upgrade the postgres client library (libpq) for SNI support or pass the endpoint ID (first part of the domain name) as a parameter: '?options=endpoint%3D<endpoint-id>'. See more at https://neon.tech/sni

Error connecting to the database: 'NoneType' object has no attribute 'execute'


AttributeError: 'NoneType' object has no attribute 'close'

# create_table

In [ ]:
new_db_obj.create_table(schema_name="ml",table_name="weather",primary_key_name="id",columns_datatype_dict= {"city_name":"VARCHAR(100)","temperature": "DECIMAL","humidity": "INTEGER","pressure": "DECIMAL", "weather_description" :"TEXT","timestamp": "TIMESTAMP"})

Database connection established successfully.
Table created successfully.


# insert_entry

In [ ]:
value_dict = get_weather_data("Chennai")

In [ ]:
new_db_obj.insert_entry(schema_name="ml",table_name="weather",columns_value_dict=value_dict)

Database connection established successfully.
Entry inserted successfully.


In [ ]:
new_db_obj.fetch_data("Select * from ml.weather")

Database connection established successfully.


[(1,
  'Chennai',
  Decimal('27.2'),
  74,
  Decimal('1011.0'),
  'Partly cloudy',
  datetime.datetime(2025, 2, 15, 15, 25, 2, 365574))]

In [ ]:
value_dict = get_weather_data("Bristol")

In [ ]:
new_db_obj.insert_entry(schema_name="ml",table_name="weather",columns_value_dict=value_dict)

Database connection established successfully.
Entry inserted successfully.


In [ ]:
new_db_obj.fetch_data("Select * from ml.weather")

Database connection established successfully.


[(1,
  'Chennai',
  Decimal('27.2'),
  74,
  Decimal('1011.0'),
  'Partly cloudy',
  datetime.datetime(2025, 2, 15, 15, 25, 2, 365574)),
 (2,
  'Bristol',
  Decimal('6.4'),
  93,
  Decimal('1016.0'),
  'Overcast',
  datetime.datetime(2025, 2, 15, 15, 31, 24, 90613))]

# delete_entry

In [ ]:
new_db_obj.fetch_data("Select * from ml.weather")

Database connection established successfully.


[(2,
  'Bristol',
  Decimal('6.4'),
  93,
  Decimal('1016.0'),
  'Overcast',
  datetime.datetime(2025, 2, 15, 15, 31, 24, 90613)),
 (1,
  'Liverpool',
  Decimal('4.0'),
  93,
  Decimal('1017.0'),
  'Light rain',
  datetime.datetime(2025, 2, 15, 15, 37, 7, 171901))]

In [ ]:
new_db_obj.delete_entry(schema_name="ml",table_name="weather",primary_key_name="id",primary_key_value="1")

Database connection established successfully.
Entry deleted successfully.


In [ ]:
new_db_obj.fetch_data("Select * from ml.weather")

Database connection established successfully.


[(2,
  'Bristol',
  Decimal('6.4'),
  93,
  Decimal('1016.0'),
  'Overcast',
  datetime.datetime(2025, 2, 15, 15, 31, 24, 90613))]

# execute_query

In [ ]:
new_db_obj.fetch_data("Select * from ml.weather")

Database connection established successfully.


[(1,
  'Chennai',
  Decimal('27.2'),
  74,
  Decimal('1011.0'),
  'Partly cloudy',
  datetime.datetime(2025, 2, 15, 15, 25, 2, 365574)),
 (2,
  'Bristol',
  Decimal('6.4'),
  93,
  Decimal('1016.0'),
  'Overcast',
  datetime.datetime(2025, 2, 15, 15, 31, 24, 90613))]

In [ ]:
value_dict = get_weather_data("liverpool")

In [ ]:
new_db_obj.execute_query("UPDATE ml.weather SET city_name = 'Liverpool', temperature = 4.0, humidity = 93, pressure = 1017.0, weather_description = 'Light rain', timestamp = '2025-02-15 15:37:07.171901' WHERE id = 1;")

Database connection established successfully.
Query executed successfully.


In [ ]:
new_db_obj.fetch_data("Select * from ml.weather")

Database connection established successfully.


[(2,
  'Bristol',
  Decimal('6.4'),
  93,
  Decimal('1016.0'),
  'Overcast',
  datetime.datetime(2025, 2, 15, 15, 31, 24, 90613)),
 (1,
  'Liverpool',
  Decimal('4.0'),
  93,
  Decimal('1017.0'),
  'Light rain',
  datetime.datetime(2025, 2, 15, 15, 37, 7, 171901))]